# Relating two vehicles that never mention each other

`simple-demo.ipynb` builds the graph and asks it questions. This notebook is about the
one question that graph could not answer: *what in Apollo plays the role of the drone's
X?*

No Apollo `.sysml` file mentions a drone, so the extraction finds no relation between
the two models -- correctly, because there is none to find. Extraction reports what the
text supports, and a resemblance between two vehicles is not in either text. The
`analogy` build step adds the missing edges.

It does not invent an algorithm to do it. autograph already answers "which of these
resemble each other" in `corpus_graph.similarity_finding.SimilarityFinder` -- semantic
search and BM25 over one corpus, fused with reciprocal rank. That class runs unmodified
against the local container. Two things were arranged around it: the corpus it is handed
is one document per **entity** rather than per file, and its `module_doc_ids` restriction
is passed the *other* models, so the only edges it can build are the ones that cross.

The step got cheaper when the parser went. It used to describe every element itself and
embed the result; now the descriptions and the vectors are the ones extraction already
wrote, so `analogy` buys no embeddings and simply reads the Entities collection.

In [1]:
import logging

from sysml import config, nl

logging.disable(logging.INFO)  # the services narrate every step; keep just the answers

db = config.db()

CROSS_MODEL = f'''
FOR r IN {config.RELATIONS}
  LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
  FILTER a.models != null AND b.models != null
  FILTER LENGTH(INTERSECTION(a.models, b.models)) == 0
  COLLECT kind = r.type WITH COUNT INTO n
  RETURN {{kind, n}}'''

print("edges joining two entities with no model in common:")
for row in db.aql.execute(CROSS_MODEL):
    print(f"  {row['n']:>4}  {row['kind']}")

edges joining two entities with no model in common:


    97  SIMILAR_TO


`SIMILAR_TO` is the only one, and that is the point: every other edge in the graph came
out of a piece of text, and no piece of text crosses a model boundary.

It is autograph's label, imported from `corpus_graph.naming`, deliberately not the
importer's. An analogy is not something a SysML file states, so it must not become a
`RELATED_TO` carrying a `relationship_type` -- that field means "the source says this",
and a computed resemblance sitting in it would land in every count that groups by it.

In [2]:
STRONGEST = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "{config.SIMILAR_TO}"
  LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
  SORT r.cosine DESC
  LIMIT 14
  RETURN {{a: a.entity_name, am: a.models[0], b: b.entity_name, bm: b.models[0],
           role: r.analogy_role, cosine: r.cosine}}'''

for r in db.aql.execute(STRONGEST):
    print(f"{r['cosine']:.3f}  {r['role']:<11} "
          f"{r['a']} ({r['am']})  ~  {r['b']} ({r['bm']})")

0.822  part        DRONEBATTERY_PARTS_DRONEBATTERY (DroneModelLogical)  ~  DRONE_BATTERY (Drone_BaseArchitecture)
0.808  part        BATTERY (Drone_BaseArchitecture)  ~  BATTERY (DroneModelLogical)
0.795  part        DRONE_DRONE (DroneModelLogical)  ~  DRONE_BASEARCHITECTURE_DRONE (Drone_BaseArchitecture)
0.786  part        DRONE_SYSTEMARCHITECTURE_DRONE (Drone_BaseArchitecture)  ~  DRONE_DRONE (DroneModelLogical)
0.765  part        DRONE_BATTERY (Drone_BaseArchitecture)  ~  BATTERY (DroneModelLogical)
0.734  part        DRONE_BASEARCHITECTURE_DRONE (Drone_BaseArchitecture)  ~  DRONEENGINE_PARTS_DRONEENGINE (DroneModelLogical)
0.728  part        DRONEENGINE_PARTS_DRONEENGINE (DroneModelLogical)  ~  DRONE_SYSTEMARCHITECTURE_DRONE (Drone_BaseArchitecture)
0.728  part        DRONE_BASEARCHITECTURE_DRONE (Drone_BaseArchitecture)  ~  DRONEBODY_PARTS_DRONEBODY (DroneModelLogical)
0.727  part        DRONE_SYSTEMARCHITECTURE_DRONE (Drone_BaseArchitecture)  ~  DRONEBODY_PARTS_DRONEBODY (DroneMo

Read that as a similarity layer and not as an oracle. Further down the same list are
`engine2 ~ five J-2 engines` and `powerManagementModule ~ powerGenerationAndDistribution`,
which are the correspondences an engineer would draw; also on it are
`connection ~ capabilityToGoalDerivation`, which is noise, and
`powerManagementModule ~ command/service module`, which matched on the word "module".

The top of the list is dominated by Drone_BaseArchitecture against DroneModelLogical, because those two
files describe the same vehicle twice and so genuinely do resemble each other more than
either resembles Apollo. Two caps keep any one element from taking over: at most three
counterparts per element, and at most two elements pointing at any one counterpart.

Now ask it in English. AQLizer writes the traversal.

In [3]:
nl.instance().ask(
    "What does the drone's powerManagementModule correspond to in the Apollo model?").show()

Q  What does the drone's powerManagementModule correspond to in the Apollo model?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER CONTAINS(LOWER(e.entity_name), "powermanagementmodule") 
     AND "DroneModelLogical" IN e.models
     FOR v, r IN 1..1 ANY e sysml_Relations
       FILTER r.type == 'SIMILAR_TO'
       FILTER "apollo-11-sysml-v2" IN v.models
       RETURN {element: e.entity_name, counterpart: v.entity_name, role: r.analogy_role, cosine: r.cosine, why: r.description}

rows (3, first 3)
   {"element": "POWERMANAGEMENTMODULE", "counterpart": "POWERGENERATIONANDDISTRIBUTION", "role": "part", "cosine": 0.5510419118302345, "why": "POWERMANAGEMENTMODULE in the DroneModelLogical model plays a role like POWERGENERATIONANDDISTRIBUTION in the apollo-11-sysml-v2 model: both are part elements an
   {"element": "POWERMANAGEMENTMODULE", "counterpart": "POWERPROVIDER", "role": "part", "cosine": 0.5564601816956668, "why": "POWERMANAGEMENTMODULE in the Dro

The retrieval path needed no changes at all. The local retriever expands over any edge
touching an entity it matched, so an analogy edge enters the context on its own, and its
`description` was written as a sentence for exactly that reason.

In [4]:
(await nl.retriever().ask_async(
    "What in the Apollo model plays a role like the drone's power management module?")).show()

Q  What in the Apollo model plays a role like the drone's power management module?

retrieved  28 documents, 81 edges, 80,399 chars of context

cited (14, first 6)
   {"cite": 1, "source": "models/DroneModelLogical.sysml"}
   {"cite": 10, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 11, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 12, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 13, "source": "models/apollo-11-sysml-v2/Technical/AstronautsPackage.sysml"}
   {"cite": 14, "source": "models/apollo-11-sysml-v2/Program/ProgramPackage.sysml"}

A  ## Comparison of Power Management Roles in Apollo and Drone Models

In the context of the Apollo model and the Forest Fire Observation Drone model, several components play roles related to managing or distributing power. Below is a comparison based on the context provided:

### Drone Power Manag

The more useful question for a systems engineer is the negative one. An analogy layer
that only reports matches is a search box; what it is actually good for is finding the
parts of one vehicle that nothing in the other resembles.

In [5]:
nl.instance().ask(
    "Which DroneModelLogical entities of type part have no SIMILAR_TO edge at all? "
    "List the first eight with the files they appear in.").show(row_limit=8)

Q  Which DroneModelLogical entities of type part have no SIMILAR_TO edge at all? List the first eight with the files they appear in.

AQL
   WITH sysml_Chunks, sysml_Documents, sysml_Entities, sysml_Communities
   FOR e IN sysml_Entities
     FILTER 'DroneModelLogical' IN e.models AND e.entity_type == 'part'
     LET analogues = LENGTH(
       FOR v, r IN 1..1 ANY e sysml_Relations
         FILTER r.type == 'SIMILAR_TO'
         RETURN 1)
     FILTER analogues == 0
     LIMIT 8
     RETURN {name: e.entity_name, files: e.files}

rows (8, first 8)
   {"name": "DRONEENGINEVARIATION", "files": ["DroneModelLogical.sysml"]}
   {"name": "ARMS", "files": ["DroneModelLogical.sysml"]}
   {"name": "FORESTFIREOBSERVATIONDRONE_FORESTFIREOBSERVATIONDRONE", "files": ["DroneModelLogical.sysml"]}
   {"name": "CHARGER", "files": ["DroneModelLogical.sysml"]}
   {"name": "DRONECONTROLUNIT", "files": ["DroneModelLogical.sysml"]}
   {"name": "SAFETYMODULE", "files": ["DroneModelLogical.sysml"]}
   {"name": 

## What this does and does not establish

It relates two models that never reference each other, it does so with autograph's own
similarity code rather than a bespoke one, and both read paths reach it -- one with the
AQL shown, the other with citations back to a source file.

What it is not: these are resemblances between *descriptions*, scored and thresholded,
and the descriptions are now themselves LLM-written. Two layers of judgement sit under
every edge, and nothing in the layer knows which of them is doing the work. The cosine
is on every edge so a reader can judge, and the floor (0.55) and both caps are constants
at the top of `sysml/pipeline/analogy.py`.

One thing worth knowing before pointing this at a bigger corpus: autograph's corpus
layer truncates each document to `CHUNK_MAX_CHARS` -- 1200 tokens x 4 = 4,800 characters.
The largest file here is 58,902 bytes, so a document-level run would have compared 8% of
it. Raising `chunk_size` helps and does not solve it: the embedding model's context ends
at 8,192 tokens, and one vector for a 58 KB file of 296 requirements does not resemble
anything in particular no matter how much of it was read. Comparing entities sidesteps
both -- every entity description is far inside the bound.